In [1]:
# Sprint 1 - importando as bibliotecas e carregando a base
import pandas as pd
import csv

df = pd.read_csv("Base Varejo.csv", sep=";")

print("Numero de registros:", df.shape[0])
print("Numero de colunas:", df.shape[1])
print(df.columns.tolist())
print(df.dtypes)
df.head()

Numero de registros: 830000
Numero de colunas: 14
['DATA', 'CO_ID', 'CL_ID', 'CL_GENERO', 'CL_EC', 'CL_FHL', 'CL_SEG', 'PR_ID', 'PR_CAT', 'PR_NOME', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13']
DATA            object
CO_ID            int64
CL_ID            int64
CL_GENERO       object
CL_EC            int64
CL_FHL           int64
CL_SEG          object
PR_ID            int64
PR_CAT          object
PR_NOME         object
Unnamed: 10    float64
Unnamed: 11    float64
Unnamed: 12    float64
Unnamed: 13    float64
dtype: object


,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,01/02/2019,1000,534,M,4,1,C,67,BEBIDAS,REFRIGERANTE GUARANA,NaN,NaN,NaN,NaN
1,01/02/2019,1000,534,M,4,1,C,70,BEBIDAS,REFRIGERANTE OUTROS,NaN,NaN,NaN,NaN
2,01/02/2019,1000,534,M,4,1,C,178,HIGIENE,LENCO UMEDECIDO,NaN,NaN,NaN,NaN
3,01/02/2019,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI,NaN,NaN,NaN,NaN
4,01/02/2019,1000,534,M,4,1,C,175,LIMPEZA,LIMPADOR MULTIUSO,NaN,NaN,NaN,NaN


In [2]:
# leitura estruturada com csv.DictReader (cada linha vira um dicionario)
with open("Base Varejo.csv", encoding="latin-1") as f:
    leitor = csv.DictReader(f, delimiter=";")
    linhas = list(leitor)

print("Linhas lidas com DictReader:", len(linhas))
print("Primeira linha:", linhas[0])

Linhas lidas com DictReader: 830000
Primeira linha: {'DATA': '01/02/2019', 'CO_ID': '1000', 'CL_ID': '534', 'CL_GENERO': 'M', 'CL_EC': '4', 'CL_FHL': '1', 'CL_SEG': 'C', 'PR_ID': '67', 'PR_CAT': 'BEBIDAS', 'PR_NOME': 'REFRIGERANTE GUARANA', '': ''}


In [3]:
# verificando problemas na base
print("Nulos por coluna:")
print(df.isnull().sum())

print("Linhas duplicadas:", df.duplicated().sum())

print("Categorias:")
print(df["PR_CAT"].value_counts(dropna=False))

print("Exemplo de datas:", df["DATA"].head().tolist())

Nulos por coluna:
DATA                0
CO_ID               0
CL_ID               0
CL_GENERO           0
CL_EC               0
CL_FHL              0
CL_SEG              0
PR_ID               0
PR_CAT              0
PR_NOME             0
Unnamed: 10    830000
Unnamed: 11    830000
Unnamed: 12    830000
Unnamed: 13    830000
dtype: int64


Linhas duplicadas: 96553
Categorias:
PR_CAT
ALIMENTOS     434767
HIGIENE       155574
LIMPEZA       145754
BEBIDAS        43299
PET            32399
ACESSORIOS     14557
#N/D            3650
Name: count, dtype: int64
Exemplo de datas: ['01/02/2019', '01/02/2019', '01/02/2019', '01/02/2019', '01/02/2019']


In [4]:
# Sprint 2 - limpeza de texto: maiusculas, tira espacios extras com regex
df = df.loc[:, ~df.columns.str.contains("Unnamed")]   # remove colunas vazias

df["PR_NOME"] = df["PR_NOME"].astype(str).str.upper().str.replace(r"\s+", " ", regex=True).str.strip()
df["PR_CAT"] = df["PR_CAT"].astype(str).str.upper().str.strip()

# converte numero de filhos para tipo numerico
df["CL_FHL"] = pd.to_numeric(df["CL_FHL"], errors="coerce")
print(df.dtypes)

DATA         object
CO_ID         int64
CL_ID         int64
CL_GENERO    object
CL_EC         int64
CL_FHL        int64
CL_SEG       object
PR_ID         int64
PR_CAT       object
PR_NOME      object
dtype: object


In [5]:
# preenchendo categorias vazias com "Sem Categoria" usando if/else
def arrumar_categoria(cat):
    if pd.isna(cat) or cat in ["", "NAN", "#N/D"]:
        return "Sem Categoria"
    else:
        return cat

df["PR_CAT"] = df["PR_CAT"].apply(arrumar_categoria)
print(df["PR_CAT"].value_counts())

# dimensoes fisicas (peso, altura, largura...): procurei e a base NAO tem essas colunas,
# entao nao ha nulos fisicos para tratar. Se tivesse, eu preencheria com a mediana
# da categoria, porque a mediana nao e afetada por valores extremos.
colunas_fisicas = [c for c in df.columns if any(p in c.upper() for p in ["PESO", "ALT", "LARG", "COMP"])]
print("Colunas fisicas encontradas:", colunas_fisicas)

# numero de filhos nulo: preencho com a mediana (se houver)
if df["CL_FHL"].isnull().sum() > 0:
    df["CL_FHL"] = df["CL_FHL"].fillna(df["CL_FHL"].median())
else:
    print("CL_FHL sem nulos")

PR_CAT
ALIMENTOS        434767
HIGIENE          155574
LIMPEZA          145754
BEBIDAS           43299
PET               32399
ACESSORIOS        14557
Sem Categoria      3650
Name: count, dtype: int64
Colunas fisicas encontradas: []
CL_FHL sem nulos


In [6]:
# removendo duplicatas exatas
# hipotese adotada: linhas identicas representam registros duplicados
# limitacao: sem quantidade ou identificador do item, nao e possivel
# descartar que sejam unidades repetidas do mesmo produto na mesma compra
antes = len(df)
df = df.drop_duplicates()
print("Duplicatas removidas:", antes - len(df))

Duplicatas removidas: 96553


In [7]:
# converte datas invalidas para NaT, sem interromper o programa
df["DATA"] = pd.to_datetime(df["DATA"], format="%d/%m/%Y", errors="coerce")

print("Datas invalidas ou ausentes:", df["DATA"].isna().sum())

# removo registros sem data valida porque nao podem participar das analises por periodo
df = df.dropna(subset=["DATA"]).copy()

# regra adotada: cada compra deve ter identificador,
# exatamente um cliente identificado e exatamente uma data
regra = df.groupby("CO_ID").agg(
    clientes=("CL_ID", "nunique"),
    datas=("DATA", "nunique"),
    cliente_ausente=("CL_ID", lambda coluna: coluna.isna().any())
)
invalidas = regra[
    (regra["clientes"] != 1)
    | (regra["datas"] != 1)
    | regra["cliente_ausente"]
]
mascara_invalidos = (
    df["CO_ID"].isna()
    | df["CO_ID"].isin(invalidas.index)
)
df_invalidos = df.loc[mascara_invalidos].copy()
df = df.loc[~mascara_invalidos].copy()

print("Compras identificadas que quebram a regra:", len(invalidas))
print("Registros separados pela regra do CO_ID:", len(df_invalidos))

Datas invalidas ou ausentes: 0


Compras identificadas que quebram a regra: 0
Registros separados pela regra do CO_ID: 0


In [8]:
# estatisticas do numero de filhos (1 linha por cliente para nao contar repetido)
filhos = df.drop_duplicates("CL_ID")["CL_FHL"]

print("Contagem:", filhos.count())
print("Media:", round(filhos.mean(), 2))
print("Mediana:", filhos.median())
print("Desvio padrao:", round(filhos.std(), 2))
modas = filhos.mode()
print("Moda:", modas.tolist() if not modas.empty else "Sem dados")
print("Minimo:", filhos.min())
print("Maximo:", filhos.max())
print("Clientes sem filhos:", (filhos == 0).sum())

Contagem: 1000
Media: 1.14
Mediana: 0.0
Desvio padrao: 1.41
Moda: [0]
Minimo: 0
Maximo: 4
Clientes sem filhos: 527


In [9]:
# agrupamento 1: genero x compras e registros de produtos
print(df.groupby("CL_GENERO").agg(compras=("CO_ID", "nunique"), registros_produtos=("PR_ID", "count")))

# agrupamento 2: quantidade de registros por categoria
print(df.groupby("PR_CAT").size().sort_values(ascending=False))

# agrupamento 3: pivot categoria x segmento
print(pd.pivot_table(df, index="PR_CAT", columns="CL_SEG", values="PR_ID", aggfunc="count"))

# agrupamento 4: compras por mes
print(df.groupby(df["DATA"].dt.to_period("M"))["CO_ID"].nunique())

           compras  registros_produtos
CL_GENERO                             
F             9615              382427
M             8856              351020
PR_CAT
ALIMENTOS        384197
HIGIENE          137702
LIMPEZA          128632
BEBIDAS           38264
PET               28553
ACESSORIOS        12871
Sem Categoria      3228
dtype: int64
CL_SEG             A       B       C
PR_CAT                              
ACESSORIOS      1043    8165    3663
ALIMENTOS      31118  245501  107578
BEBIDAS         3166   24363   10735
HIGIENE        11326   87943   38433
LIMPEZA        10460   82063   36109
PET             2303   18406    7844
Sem Categoria    261    2064     903
DATA
2019-01    171
2019-02    402
2019-03    257
2019-04    410
2019-05    711
2019-06    493
2019-07    274
2019-08    420
2019-09    436
2019-10    242
2019-11    151
2019-12    516
2020-01    506
2020-02    434
2020-03    439
2020-04    387
2020-05    366
2020-06    264
2020-07    215
2020-08    384
2020-09    471
202

In [10]:
# relatorio final
print("===== RELATORIO FINAL =====")
print("Registros finais:", len(df))
print("Clientes:", df["CL_ID"].nunique())
print("Compras:", df["CO_ID"].nunique())
print("Produtos:", df["PR_ID"].nunique())
print("Categorias:", df["PR_CAT"].nunique())
print("Registros separados pela regra do CO_ID:", len(df_invalidos))

df.to_csv("df_limpo.csv", index=False)
print("df_limpo.csv salvo")

===== RELATORIO FINAL =====
Registros finais: 733447
Clientes: 1000
Compras: 18471
Produtos: 229


Categorias: 7
Registros separados pela regra do CO_ID: 0


df_limpo.csv salvo
